In [ ]:
#Primer importem les llibreries
import pandas as pd
import os
from transformers import AutoModelForMaskedLM, AutoModelForCausalLM
from transformers import AutoTokenizer, FillMaskPipeline
from pprint import pprint
import torch
import numpy as np

In [ ]:
#Llegim les dades
df = pd.read_csv("dataset_1.csv")

#Això es podria haver fet manualment, però afegim noves columnes on el context s'ha substituït per el neologisme, el distractor i les altres opcions errònies
df.columns = df.columns.str.strip() #per treure els espais als noms de les columnes, em sortia error
df['context_neologisme'] = df.apply(lambda row: row['context'].replace('<mask>', row['neologisme']), axis=1)
df['context_distractor'] = df.apply(lambda row: row['context'].replace('<mask>', row['distractor']), axis=1)
df['context_incorrecte_1'] = df.apply(lambda row: row['context'].replace('<mask>', row['incorrecte 1']), axis=1)
df['context_incorrecte_2'] = df.apply(lambda row: row['context'].replace('<mask>', row['incorrecte 2']), axis=1)
df.head()

,neologisme,influencia,context,distractor,incorrecte 1,incorrecte 2,context_neologisme,context_distractor,context_incorrecte_1,context_incorrecte_2
0,quinto,no,"A la festa major, el cambrer li va portar un <...",got,glop,refresc,"A la festa major, el cambrer li va portar un q...","A la festa major, el cambrer li va portar un g...","A la festa major, el cambrer li va portar un g...","A la festa major, el cambrer li va portar un r..."
1,fangoteràpia,no,La clínica ofereix tractaments de benestar com...,fisioteràpia,estètica,vapor,La clínica ofereix tractaments de benestar com...,La clínica ofereix tractaments de benestar com...,La clínica ofereix tractaments de benestar com...,La clínica ofereix tractaments de benestar com...
2,velcro,no,Les bambes dels nens petits porten <mask> per ...,cremallera,cordons,pintura,Les bambes dels nens petits porten velcro per ...,Les bambes dels nens petits porten cremallera ...,Les bambes dels nens petits porten cordons per...,Les bambes dels nens petits porten pintura per...
3,mutar,no,Els científics van observar que el virus podia...,evolucionar,desaparèixer,menjar,Els científics van observar que el virus podia...,Els científics van observar que el virus podia...,Els científics van observar que el virus podia...,Els científics van observar que el virus podia...
4,prepagament,no,Per reservar l'apartament de vacances cal fer ...,crèdit,regal,transport,Per reservar l'apartament de vacances cal fer ...,Per reservar l'apartament de vacances cal fer ...,Per reservar l'apartament de vacances cal fer ...,Per reservar l'apartament de vacances cal fer ...


In [ ]:
#Aquí carreguem els models, els altres estan comentats perquè he anat un per un per no sobrecarregar el Google Colab
model_name = 'BSC-LT/salamandra-7b'
#model_name = 'catallama/CataLlama-v0.1-Base'
#model_name = 'facebook/xglm-7.5B'
#model_name = 'catallama/CataLlama-v0.2-Base'
#model_name = 'gplsi/Aitana-6.3B'
#model_name = 'BSC-LT/ALIA-40b'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/989 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/37.0M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

In [ ]:
#Amb aquesta funció calculem la log-likelihood (NLL) de cada oració. S'ha de repetir per a cada model
def nll(model_name, dataset, tokenizer):
  #model.eval()
  #tokenizer = AutoTokenizer.from_pretrained(model_name)
  #model = AutoModelForCausalLM.from_pretrained(model_name)  # <-- carreguem el model real
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model.to(device)
  results = []

  #iterem totes les oracions, que ara ja s'han subsituït pel neologisme, distractor, opció errònia 1 i opció errònia 2
  for index, row in dataset.iterrows():
    neologisme = row['context_neologisme']
    distractor = row['context_distractor']
    erroni_1 = row['context_incorrecte_1']
    erroni_2 = row['context_incorrecte_2']


    contexts = {
    'neologisme': neologisme,
    'distractor': distractor,
    'erroni_1': erroni_1,
    'erroni_2': erroni_2}

    row_losses = {}

    for label, sentence in contexts.items():

      inputs = tokenizer(sentence, return_tensors="pt")
      inputs = {k: v.to(device) for k, v in inputs.items()}

      with torch.no_grad():
          # En passar 'labels', el model calcula la CrossEntropyLoss internament
          # que és exactament la NLL mitjana per token.
          outputs = model(**inputs, labels=inputs["input_ids"])

      row_losses[label] = outputs.loss.item() #aquí es calcula la loss (pèrdua) de cada neologisme, distractor

    results.append(row_losses)

  return results

In [ ]:
#Aquesta secció considera totes les NLLs que produeix la funció, un total de 80 judicis per model, i tria la millor opció per cada context

losses = nll(model, df, tokenizer)  #l'output de la funció

# Troba la millor opció per cada context (la que té la loss més baixa)
millors_opcions = [min(row_loss, key=row_loss.get) for row_loss in losses]

#Ho afegeix en una nova columna
df['model_pick'] = millors_opcions

# També documentem la loss (pèrdua de la opció escullida)
picked_losses = [row_loss[pick] for row_loss, pick in zip(losses, millors_opcions)]
df['model_pick_loss'] = picked_losses

# Ho exportem a un csv, que he resumit en un altre CSV de manera manual, i que també està al repositori amb el nom 'Resultats.csv" per al primer corpus de contextos, i "Resultats_nous.csv" per al segon corpus
df.to_csv("prediccions.csv", index=False)



✅ Exported results with model picks to 'model_neologism_predictions.csv'
